## Convert Phase 1 to  script (scraping)

scripts/scrape_books.py

In [5]:
import requests
from bs4 import BeautifulSoup

def scrape_books():
    url = "https://books.toscrape.com/"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    data = []
    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        availability = book.find(
            "p", class_="instock availability"
        ).text.strip()

        data.append({
            "title": title,
            "price_gbp": price,
            "availability": availability
        })

    return data


## Convert Phase 2 to script (API)

scripts/fetch_exchange_rate.py

In [6]:
import requests

def fetch_exchange_rate():
    url = "https://open.er-api.com/v6/latest/GBP"
    response = requests.get(url)
    data = response.json()

    return data["rates"]["INR"]


## Convert Phase 3 to script (transform)

scripts/transform_data.py

In [7]:
import pandas as pd
import hashlib

def transform_data(raw_data, gbp_to_inr):
    df = pd.DataFrame(raw_data)

    # clean price
    df["price_gbp"] = (
        df["price_gbp"]
        .astype(str)
        .str.replace("Â", "", regex=False)
        .str.replace("£", "", regex=False)
        .astype(float)
    )

    # convert currency
    df["price_inr"] = df["price_gbp"] * gbp_to_inr

    # availability
    df["in_stock"] = df["availability"].str.contains("In stock")

    # price tier
    def price_tier(price):
        if price < 500:
            return "cheap"
        elif price < 1500:
            return "moderate"
        else:
            return "expensive"

    df["price_tier"] = df["price_inr"].apply(price_tier)

    # product id
    def generate_product_id(row):
        raw = f"{row['title']}_{row['price_gbp']}"
        return hashlib.md5(raw.encode()).hexdigest()

    df["product_id"] = df.apply(generate_product_id, axis=1)

    return df


## Convert Phase 4 to script (load)

scripts/load_to_db.py


In [ ]:
import psycopg2

def load_to_db(df, gbp_to_inr):
    conn = psycopg2.connect(
        host="postgres",       # Docker later
        database="pricing",
        user="airflow",
        password="airflow"
    )
    cursor = conn.cursor()

    # insert exchange rate
    cursor.execute("""
    INSERT INTO staging_exchange_rates (rate_date, gbp_to_inr)
    VALUES (CURRENT_DATE, %s)
    ON CONFLICT (rate_date) DO NOTHING;
    """, (gbp_to_inr,))

    # insert products
    for _, row in df.iterrows():
        cursor.execute("""
        INSERT INTO products (
            product_id, title,
            price_gbp, price_inr,
            in_stock, price_tier
        )
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (product_id) DO UPDATE
        SET
            price_gbp = EXCLUDED.price_gbp,
            price_inr = EXCLUDED.price_inr,
            in_stock = EXCLUDED.in_stock,
            price_tier = EXCLUDED.price_tier;
        """, (
            row["product_id"],
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["in_stock"],
            row["price_tier"]
        ))

    conn.commit()
    cursor.close()
    conn.close()


## Airflow

dags/pricing_pipeline.py

In [8]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from scripts.scrape_books import scrape_books
from scripts.fetch_exchange_rate import fetch_exchange_rate
from scripts.transform_data import transform_data
from scripts.load_to_db import load_to_db

def run_pipeline():
    raw_data = scrape_books()
    rate = fetch_exchange_rate()
    df = transform_data(raw_data, rate)
    load_to_db(df, rate)

with DAG(
    dag_id="pricing_pipeline",
    start_date=datetime(2025, 1, 1),
    schedule="0 10 * * *",
    catchup=False
) as dag:

    run = PythonOperator(
        task_id="run_pipeline",
        python_callable=run_pipeline
    )


ModuleNotFoundError: No module named 'airflow'